In [ ]:
# Group by customer and invoice, then sum the costs and quantities
invoice_df = df.groupby(['Customer ID', 'Invoice']).agg(
    total_cost=('line_cost', 'sum'),
    number_of_items=('Quantity', 'sum')
).reset_index()

invoice_df.to_csv("invoice.csv", index=False)

# Sort descending and take the head
invoice_df.sort_values(by='number_of_items', ascending=False).head(10);

# Sort ascending and take the head
bottom_10_invoices = invoice_df.sort_values(by='number_of_items', ascending=True).head(10)
bottom_10_invoices;

### Creating product table

product_table = (
    df[['StockCode', 'Description']]
    .dropna()
    .drop_duplicates(subset=['StockCode'])
    .copy()
)

product_table['Description'] = product_table['Description'].astype(str)

print(product_table.shape)

### Creating short description using product table

def clean_product_description(text):
    text = str(text).lower()

    # remove numbers and measurements
    text = re.sub(r'\b\d+\s*(cm|mm|m|inch|inches|oz|ml|l|kg|g)\b', ' ', text)
    text = re.sub(r'\b\d+\b', ' ', text)

    # remove punctuation
    text = re.sub(r'[^a-z\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # words that usually do not define category
    noise_words = {
        'set', 'pack', 'piece', 'pieces', 'small', 'large', 'medium',
        'mini', 'big', 'assorted', 'colour', 'colours', 'red', 'blue',
        'green', 'pink', 'white', 'black', 'yellow', 'purple', 'orange',
        'brown', 'silver', 'gold', 'metal', 'wooden', 'wood', 'glass',
        'ceramic', 'paper', 'plastic', 'felt', 'cotton', 'cm', 'mm',
        'round', 'square', 'heart', 'star', 'vintage', 'retro'
    }

    words = text.split()
    words = [w for w in words if w not in noise_words]

    return ' '.join(words)

product_table['short_description'] = product_table['Description'].apply(
    clean_product_description
)